# Add `eopf:origin_datetime` property to Cadip sessions

See: https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-616

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *
init_demo()
# Reload the global vars again
from resources.utils import *  

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

In [ ]:
# Create test collection
catalog_collection = "TEST_EOPF_ORIGIN_DATETIME"
create_test_collection(catalog_collection)

In [ ]:
def run_demo(cadip_collection: str, cadip_session: str):
    """Run demo for a cadip collection and session"""

    cadip_href = "http://localhost:8002" if local_mode else os.environ["RSPY_WEBSITE"]
    print(f"Get session from rs-server-cadip: {cadip_href}/cadip/collections/{cadip_collection}/items/{cadip_session}")

    cadip_item = cadip_client.get_item(cadip_collection, cadip_session)
    assert cadip_item
    print(f"""
Response from rs-server-cadip:
"id": "{cadip_item.id}"
"properties": {{
    "end_datetime":         {cadip_item.properties["end_datetime"]}
    "eopf:origin_datetime": {cadip_item.properties["eopf:origin_datetime"]}
}}
""")
    assert (
        datetime.fromisoformat(cadip_item.properties["end_datetime"]) == 
        datetime.fromisoformat(cadip_item.properties["eopf:origin_datetime"])
    )
    
    print("Stage Cadip session")
    job_status = staging_client.run_staging(cadip_item.to_dict(), catalog_collection)
    job_status = staging_client.wait_for_jobs(job_status, logger)

    catalog_href = "http://localhost:8003" if local_mode else os.environ["RSPY_WEBSITE"]
    print(f"Get session from rs-server-catalog: {catalog_href}/catalog/collections/{catalog_collection}/items/{cadip_session}")

    catalog_item = catalog_client.get_item(catalog_collection, cadip_session)
    assert catalog_item
    print(f"""
Response from rs-server-catalog:
"id": "{catalog_item.id}"
"properties": {{
    "end_datetime":         {catalog_item.properties["end_datetime"]}
    "eopf:origin_datetime": {catalog_item.properties["eopf:origin_datetime"]}
}}
""")
    assert (
        datetime.fromisoformat(cadip_item.properties["end_datetime"]) == 
        datetime.fromisoformat(catalog_item.properties["end_datetime"]) == 
        datetime.fromisoformat(catalog_item.properties["eopf:origin_datetime"])
    )


In [ ]:
# Run demo for S1A session
run_demo("sgs_sentinel1", "S1A_20200105072204051312")

In [ ]:
# Run demo for S3B session
run_demo("sgs_sentinel3", "S3B_20251010143722593812")